In [1]:

# Ejecuta esta celda una sola vez; reinicia el kernel si es necesario
import importlib, subprocess, sys
if importlib.util.find_spec('shap') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    print('shap instalado correctamente. Continua con la siguiente celda.')
else:
    print('shap ya esta instalado.')

shap ya esta instalado.


In [ ]:
# ============================================================
# BLOQUE 0: IMPORTACIONES CONSOLIDADAS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Preprocesamiento
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, cross_val_predict
)

# Metricas
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, f1_score
)

# SHAP
try:
    import shap
    SHAP_DISPONIBLE = True
except ImportError:
    print('SHAP no disponible. Ejecuta primero la celda de instalacion.')
    SHAP_DISPONIBLE = False

import session_info

In [ ]:
# 1. Cargar archivos
df_tic      = pd.read_csv('Tecnologías de información y comunicación.CSV', sep=';', encoding='latin-1')
df_vivienda = pd.read_csv('Datos de la vivienda.CSV',                      sep=';', encoding='latin-1')
df_ipm      = pd.read_csv('hogares_2023_dpto.CSV',                         sep=',', encoding='latin-1')

# 2. Normalizar nombres a MAYUSCULAS
df_tic.columns      = df_tic.columns.str.upper().str.strip()
df_vivienda.columns = df_vivienda.columns.str.upper().str.strip()
df_ipm.columns      = df_ipm.columns.str.upper().str.strip()

# 3. Asegurar que DIRECTORIO sea string
for d in [df_tic, df_vivienda, df_ipm]:
    d['DIRECTORIO'] = d['DIRECTORIO'].astype(str).str.strip()

cols_vivienda = ['DIRECTORIO', 'P1_DEPARTAMENTO', 'CLASE']
if 'P1_DEPARTAMENTO' not in df_vivienda.columns:
    col_depto_viv = [c for c in df_vivienda.columns if 'DEP' in c][0]
    cols_vivienda[1] = col_depto_viv

df_temp  = pd.merge(df_tic,  df_vivienda[cols_vivienda],          on='DIRECTORIO', how='inner')
df_final = pd.merge(df_temp, df_ipm[['DIRECTORIO', 'POBRE']],     on='DIRECTORIO', how='inner')

print(f"Filas finales: {df_final.shape[0]}")
print(f"Columna departamento presente: {'P1_DEPARTAMENTO' in df_final.columns}")

In [ ]:
diccionario_nombres = {
    'P1_DEPARTAMENTO': 'Departamento',
    'CLASE': 'Zona Urbano Rural',
    'P765S1': 'Computador de Escritorio',
    'P765S2': 'Computador Portatil',
    'P765S3': 'Tableta',
    'P765S4': 'Telefono Celular',
    'P765S5': 'Consola para juegos',
    'P765S6': 'Televisor inteligente',
    'P765S7': 'Reproductor de musica',
    'P1929':  'Cual es la razon por la que no utiliza internet',
    'P1710S1':  'Hab Procesador Texto',
    'P1710S2':  'Hab Mover Archivos',
    'P1710S3':  'Hab Enviar Correos',
    'P1710S4':  'Hab Conectar Equipos',
    'P1710S5':  'Hab Excel Formulas',
    'P1710S6':  'Hab Presentaciones',
    'P1710S7':  'Hab Transferir Archivos',
    'P1710S8':  'Hab Instalar Software',
    'P1710S9':  'Hab Programacion',
    'P1710S10': 'Hab Seguridad Digital',
    'P1710S11': 'Hab Privacidad Datos',
    'P1710S12': 'Hab Verificar Noticias',
    'P1083S1':  'Uso Int Informacion',
    'P1083S2':  'Uso Int Correos Electronicos',
    'P1083S3':  'Uso Int Redes Sociales',
    'P1083S4':  'Uso Int Compras',
    'P1083S5':  'Uso Int Banca Electronica',
    'P1083S6':  'Uso Int Educacion',
    'P1083S7':  'Uso Int Tramites Gobierno',
    'P1083S8':  'Uso Int Descargas',
    'P1083S9':  'Uso Int Medios de Comunicacion',
    'P1083S10': 'Uso Int Audiovisual',
    'P1083S12': 'Uso Int Buscar Trabajo',
    'P1083S13': 'Uso Int Nube',
    'P1083S14': 'Uso Int Vender Productos',
    'P1083S15': 'Uso Int Trabajar',
    'P1083S16': 'Uso Int Llamadas',
    'P1083S17': 'Uso Int Publicar y Calificar Servicios',
    'P1083S18': 'Uso Int Post',
    'P1080S1': 'Uso Tel Lamadas Personales',
    'P1080S2': 'Uso Tel Lamadas Laborales',
    'P1080S3': 'Uso Tel Mensajes de Texto',
    'P1080S4': 'Uso Tel Navegacion en Internet',
    'P1910': 'Frec Uso PC Escritorio',
    'P1911': 'Frec Uso Laptop',
    'P1912': 'Frec Uso Tableta',
    'P803':   'Pago Servicio de Telefonia Celular',
    'P803S1': 'Valor Pagado Servicio Telefonia',
    'P805S1': 'Uso Radio Entretenimiento',
    'P805S2': 'Uso Radio Noticias',
    'P805S3': 'Uso Radio Entretenimiento2',
    'P5504':  'Usa Telefono Celular Sin Ser Propio',
    'P1082':  'Tiene Celular',
    'P1082S1': 'Tiene Celular Basico',
    'P1082S2': 'Tiene Smartphone',
    'P1085S1': 'Acceso Int en Hogar',
    'P1085S2': 'Acceso Int en trabajo',
    'P1085S3': 'Acceso Int en Institucion Educativa',
    'P1085S4': 'Acceso Int Publico Gratis',
    'P1085S5': 'Acceso Int Publico Costo',
    'P1085S6': 'Acceso Int Vivienta de Otra Persona',
    'P1084':  'Frecuencia Uso Int',
}

df_final = df_final.rename(columns=diccionario_nombres)
identificadores_finales = ['DIRECTORIO', 'Departamento', 'POBRE']
X_columns = [c for c in df_final.columns if c not in identificadores_finales]
print(f"Variables renombradas: {len(diccionario_nombres)} | Predictores candidatos: {len(X_columns)}")

In [ ]:
# ============================================================
# BLOQUE 3: LIMPIEZA Y TRANSFORMACION (SIN IMPUTACION)
# Correccion: la imputacion ocurre DESPUES del split para evitar
# fuga de datos (data leakage).
# ============================================================

# 1. Eliminar columnas irrelevantes
cols_a_borrar = [
    'SECUENCIA_ENCUESTA', 'SECUENCIA_P', 'ORDEN', 'FEX_C', 'PERIODO',
    'P805', 'P805S4', 'P805S5', 'P1080', 'P1080S5', 'P1080S6',
    'P5505', 'P5505S1', 'P1085S8', 'P1083', 'P765S8', 'P769', 'P1710',
    'P1083S11', 'P804', 'P5504S1', 'P5504S2', 'P5505S2', 'P1085', 'P765', 'P1085S7'
]
df_final = df_final.drop(columns=cols_a_borrar, errors='ignore')

# 2. Asegurar nombre del departamento
for col in df_final.columns:
    if 'DEPARTAMENTO' in col.upper():
        df_final = df_final.rename(columns={col: 'Departamento'})
        break

# 3. Definicion de roles
identificadores  = ['DIRECTORIO', 'Departamento', 'POBRE']
df_final = df_final.loc[:, ~df_final.columns.duplicated()].copy()
X_columns = [c for c in df_final.columns if c not in identificadores]

# 4. Transformacion de variables de frecuencia (regla de dominio: sin dato = 0)
mapeo_intensidad = {1: 4, 2: 3, 3: 2, 4: 1, 5: 0}
columnas_frecuencia = ['Frec Uso PC Escritorio', 'Frec Uso Laptop', 'Frec Uso Tableta']
for col in columnas_frecuencia:
    if col in df_final.columns:
        col_data = df_final[col]
        if isinstance(col_data, pd.DataFrame):
            col_data = col_data.iloc[:, 0]
        # fillna(0) es una regla de dominio (no usa = intensidad 0), no imputacion estadistica
        df_final[f'Intensidad {col}'] = col_data.map(mapeo_intensidad).fillna(0)

# 5. Transformacion de razon de no uso de internet
col_razon = 'Cual es la razon por la que no utiliza internet'
if col_razon in df_final.columns:
    razon_data = df_final[col_razon]
    if isinstance(razon_data, pd.DataFrame):
        razon_data = razon_data.iloc[:, 0]
    df_final['Brecha de Internet por Costo']     = (razon_data == 1).astype(int)
    df_final['Brecha de Internet por Cobertura'] = (razon_data == 4).astype(int)
    df_final['Brecha de Internet por Habilidad'] = (razon_data == 3).astype(int)

cols_para_borrar = [col_razon] + columnas_frecuencia
df_final = df_final.drop(columns=cols_para_borrar, errors='ignore')
X_columns = [c for c in df_final.columns if c not in identificadores]

# 6. Conversion a numerico y recodificacion {1,2} -> {1,0}
for col in X_columns:
    serie = df_final[col]
    if isinstance(serie, pd.DataFrame):
        serie = serie.iloc[:, 0]
    df_final[col] = pd.to_numeric(serie, errors='coerce')
    vals = set(df_final[col].dropna().unique())
    if vals == {1, 2}:
        df_final[col] = df_final[col].replace(2, 0)

# 7. Excluir variables con varianza cero
# IMPORTANTE: el chequeo simula la imputacion que se hara despues del split:
#   - binarias (0/1): NaN -> 0  (no tiene el dispositivo = 0)
#   - ordinales       : NaN -> mediana de la columna
# Sin esto, columnas {1.0, NaN} parecen constantes y se eliminarian por error.
def std_post_imputation(s):
    vals = set(s.dropna().unique())
    if vals.issubset({0.0, 1.0}):
        return s.fillna(0).std()
    return s.fillna(s.median()).std()

cero_var = [c for c in X_columns if std_post_imputation(df_final[c]) == 0]
X_columns_modelo = [c for c in X_columns if c not in cero_var]
if cero_var:
    print(f"Variables con varianza cero excluidas: {cero_var}")

# 8. Definir X e y
X = df_final[X_columns_modelo].copy()
y = df_final['POBRE'].astype(int).copy()

print(f"Predictores: {len(X_columns_modelo)} | Registros: {len(X):,}")
print(f"Distribucion POBRE -> 0: {(y==0).sum():,} ({(y==0).mean()*100:.1f}%)  |  1: {(y==1).sum():,} ({(y==1).mean()*100:.1f}%)")

In [ ]:
# ============================================================
# BLOQUE 4: DIVISION TRAIN / TEST  —  CORRECCION CRITICA
# El split se realiza ANTES de cualquier imputacion estadistica.
# stratify=y garantiza la misma proporcion de pobres en ambos conjuntos.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 52)
print("DIVISION TRAIN / TEST")
print("=" * 52)
print(f"Train : {X_train.shape[0]:>7,} registros  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]:>7,}  registros  ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"")
print(f"Proporcion POBRE en train : {y_train.mean():.4f}")
print(f"Proporcion POBRE en test  : {y_test.mean():.4f}")
print(f"(Proporciones iguales gracias a stratify=y)")
print("=" * 52)

In [ ]:
# ============================================================
# BLOQUE 5: PREPARACION DEL CONJUNTO DE ENTRENAMIENTO PARA EDA
# Todo el analisis exploratorio se hace SOLO sobre datos de train.
# El test set queda sellado hasta la evaluacion final.
# ============================================================

# Snapshot de nulos ANTES de imputar (para heatmap y filtros de datos reales)
df_nulos_snapshot = X_train.isnull().copy()
print(f"Nulos totales en train (pre-imputacion): {df_nulos_snapshot.sum().sum():,}")

# Imputar X_train PARA EDA usando estadisticas del propio train (sin data leakage):
#   - binarias (0/1): NaN -> 0  (no tiene el dispositivo = 0)
#   - ordinales      : NaN -> mediana del train
_bin_cols_eda = [c for c in X_train.columns
                 if set(X_train[c].dropna().unique()).issubset({0.0, 1.0})]
_ord_cols_eda = [c for c in X_train.columns if c not in _bin_cols_eda]

X_train_eda = X_train.copy()
if _bin_cols_eda:
    X_train_eda[_bin_cols_eda] = (SimpleImputer(strategy='constant', fill_value=0)
                                  .fit_transform(X_train[_bin_cols_eda]))
if _ord_cols_eda:
    X_train_eda[_ord_cols_eda] = (SimpleImputer(strategy='median')
                                  .fit_transform(X_train[_ord_cols_eda]))

df_train_viz = X_train_eda.copy()
df_train_viz['POBRE']        = y_train.values
df_train_viz['Departamento'] = df_final.loc[X_train.index, 'Departamento'].values

print(f"df_train_viz: {df_train_viz.shape[0]:,} filas x {df_train_viz.shape[1]} columnas")
print(f"Nulos en df_train_viz: {df_train_viz.isnull().sum().sum()}")

In [ ]:
# ============================================================
# BLOQUE 6-A: EDA — ESTADISTICAS DESCRIPTIVAS (SOLO TRAIN)
# ============================================================

vars_interes   = ['Valor Pagado Servicio Telefonia', 'Frecuencia Uso Int', 'Tiene Smartphone', 'POBRE']
vars_existentes = [v for v in vars_interes if v in df_train_viz.columns]

print("--- ESTADISTICAS DESCRIPTIVAS (datos de entrenamiento) ---")
display(df_train_viz[vars_existentes].describe().T)

# Distribucion de POBRE
plt.figure(figsize=(9, 6))
ax = sns.countplot(x='POBRE', data=df_train_viz, palette=['#34495E', '#E74C3C'])
plt.title('Distribucion de la Pobreza Multidimensional (POBRE) — Conjunto de Entrenamiento', fontsize=14, pad=20)
plt.xticks([0, 1], ['No Pobre (0)', 'Pobre (1)'])
plt.xlabel('Condicion Socioeconomica', fontsize=12)
plt.ylabel('Numero de Registros', fontsize=12)
total = len(df_train_viz)
for p in ax.patches:
    h = p.get_height()
    ax.annotate(f'{100*h/total:.1f}%', (p.get_x() + p.get_width()/2., h),
                ha='center', va='baseline', fontsize=12, fontweight='bold',
                color='black', xytext=(0, 8), textcoords='offset points')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# Mapa de calor de nulos (pre-imputacion, solo train)
cols_snap = df_nulos_snapshot.columns[:25].tolist()
plt.figure(figsize=(15, 7))
sns.heatmap(df_nulos_snapshot[cols_snap], yticklabels=False, cbar=True, cmap='YlGnBu')
plt.title('Mapa de Calidad de Datos: Valores Nulos en Variables TIC — Train (Pre-Imputacion)', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 6-B: EDA — POBREZA POR DEPARTAMENTO (SOLO TRAIN)
# ============================================================

diccionario_deptos = {
    5:'Antioquia', 8:'Atlantico', 11:'Bogota D.C.', 13:'Bolivar', 15:'Boyaca',
    17:'Caldas', 18:'Caqueta', 19:'Cauca', 20:'Cesar', 23:'Cordoba',
    25:'Cundinamarca', 27:'Choco', 41:'Huila', 44:'La Guajira', 47:'Magdalena',
    50:'Meta', 52:'Narino', 54:'Norte de Santander', 63:'Quindio', 66:'Risaralda',
    68:'Santander', 70:'Sucre', 73:'Tolima', 76:'Valle del Cauca', 81:'Arauca',
    85:'Casanare', 86:'Putumayo', 88:'San Andres', 91:'Amazonas', 94:'Guainia',
    95:'Guaviare', 97:'Vaupes', 99:'Vichada'
}

df_train_viz['Departamento_nombre'] = df_train_viz['Departamento'].astype(float).astype('Int64').map(diccionario_deptos)
pobreza_ranking = (df_train_viz.groupby('Departamento_nombre')['POBRE']
                   .mean().sort_values(ascending=False) * 100)

plt.figure(figsize=(10, 12))
sns.barplot(x=pobreza_ranking.values, y=pobreza_ranking.index, palette='Reds_r')
plt.axvline(x=pobreza_ranking.mean(), color='black', linestyle='--',
            label=f'Promedio muestra ({pobreza_ranking.mean():.1f}%)')
plt.title('Incidencia de Pobreza Multidimensional por Departamento — Train', fontsize=15, pad=20)
plt.xlabel('Porcentaje de Hogares Pobres (%)', fontsize=12)
plt.ylabel('Departamento', fontsize=12)
plt.legend(loc='lower right')
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 6-C: EDA — BRECHA DE DISPOSITIVOS (SOLO TRAIN)
# ============================================================

dispositivos_map = {
    'Computador de Escritorio': 'PC Escritorio',
    'Computador Portatil': 'Laptop/Portatil',
    'Tableta': 'Tableta',
    'Telefono Celular': 'Celular/Smartphone',
    'Consola para juegos': 'Consola Videojuegos',
    'Televisor inteligente': 'Smart TV',
    'Reproductor de musica': 'Reproductor Musica',
}

resumen_dispositivos = []
for col_larga, nombre_corto in dispositivos_map.items():
    if col_larga in df_train_viz.columns:
        col_data = pd.to_numeric(df_train_viz[col_larga], errors='coerce').fillna(0)
        pct = col_data.groupby(df_train_viz['POBRE']).mean() * 100
        if 0 in pct.index:
            resumen_dispositivos.append({'Dispositivo': nombre_corto, 'Condicion': 'No Pobre (0)', 'Porcentaje': pct[0]})
        if 1 in pct.index:
            resumen_dispositivos.append({'Dispositivo': nombre_corto, 'Condicion': 'Pobre (1)',    'Porcentaje': pct[1]})

df_plot = pd.DataFrame(resumen_dispositivos)
plt.figure(figsize=(12, 10))
ax = sns.barplot(x='Porcentaje', y='Dispositivo', hue='Condicion', data=df_plot, palette=['#34495E', '#E74C3C'])
plt.title('Brecha de Dispositivos: Hogares con Acceso — Train', fontsize=15, pad=20)
plt.xlabel('Porcentaje de Hogares (%)', fontsize=12)
plt.xlim(0, 100)
for p in ax.patches:
    w = p.get_width()
    ax.annotate(f'{w:.1f}%', (w, p.get_y() + p.get_height()/2.),
                ha='left', va='center', fontsize=10, fontweight='bold', xytext=(5,0), textcoords='offset points')
plt.legend(title='Estado Socioeconomico')
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 6-D: EDA — USOS DE INTERNET (SOLO TRAIN)
# ============================================================

usos_interes = {
    'Uso Int Banca Electronica': 'Banca Electronica',
    'Uso Int Trabajar':          'Teletrabajo / Laboral',
    'Uso Int Educacion':         'Educacion Virtual',
    'Uso Int Tramites Gobierno': 'Tramites Estado',
    'Uso Int Vender Productos':  'Emprendimiento / Ventas',
    'Uso Int Redes Sociales':    'Redes Sociales',
}

resumen_usos = []
for col_diccionario, nombre_grafico in usos_interes.items():
    if col_diccionario in df_train_viz.columns:
        temp = pd.to_numeric(df_train_viz[col_diccionario], errors='coerce').fillna(0)
        serie_bool = (temp == 1).astype(int)
        pct = serie_bool.groupby(df_train_viz['POBRE']).mean() * 100
        if 0 in pct.index:
            resumen_usos.append({'Actividad': nombre_grafico, 'Condicion': 'No Pobre (0)', 'Porcentaje': pct[0]})
        if 1 in pct.index:
            resumen_usos.append({'Actividad': nombre_grafico, 'Condicion': 'Pobre (1)',    'Porcentaje': pct[1]})

df_usos_final = pd.DataFrame(resumen_usos)
plt.figure(figsize=(12, 8))
ax = sns.barplot(x='Porcentaje', y='Actividad', hue='Condicion', data=df_usos_final, palette=['#34495E', '#E74C3C'])
plt.title('Brecha de Apropiacion Digital: Usos del Internet — Train', fontsize=15, pad=20)
plt.xlabel('Porcentaje de Hogares que realizan la actividad (%)', fontsize=12)
plt.xlim(0, 100)
for p in ax.patches:
    w = p.get_width()
    if w > 0:
        ax.annotate(f'{w:.1f}%', (w, p.get_y() + p.get_height()/2.),
                    ha='left', va='center', fontsize=10, fontweight='bold', xytext=(5,0), textcoords='offset points')
plt.legend(title='Estado Socioeconomico')
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 6-E: EDA — CORRELACION SPEARMAN (SOLO TRAIN)
# ============================================================

# Crear dataframe con features + POBRE (solo train, con NaN aun presentes)
df_corr_base = df_train_viz[[c for c in df_train_viz.columns if c not in ['Departamento', 'Departamento_nombre']]].copy()
cols_corr = [c for c in df_corr_base.columns if df_corr_base[c].std() > 0]
print(f"Variables con varianza > 0 en train: {len(cols_corr)}")

matriz_corr = df_corr_base[cols_corr].corr(method='spearman')

umbral_pobre = 0.15
corr_con_pobre   = matriz_corr['POBRE'].drop('POBRE').abs()
cols_relevantes  = corr_con_pobre[corr_con_pobre >= umbral_pobre].index.tolist() + ['POBRE']
matriz_filtrada  = matriz_corr.loc[cols_relevantes, cols_relevantes]

print(f"Variables con |r(POBRE)| >= {umbral_pobre}: {len(cols_relevantes)-1}")

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(matriz_filtrada, dtype=bool))
sns.heatmap(matriz_filtrada, mask=mask, cmap='RdBu_r', center=0,
            annot=False, linewidths=.5, cbar_kws={'shrink': .8})
titulo = (f'Matriz de Correlacion de Spearman: Ecosistema Digital vs. Pobreza (Train)\n'
          f'(variables con |r| >= {umbral_pobre} con POBRE)')
plt.title(titulo, fontsize=16, pad=20)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

top_corr   = matriz_corr['POBRE'].drop('POBRE').abs().sort_values(ascending=False).head(10)
top_10_vars = top_corr.index.tolist()
print("\n--- TOP 10 VARIABLES MAS CORRELACIONADAS CON LA POBREZA (Spearman, Train) ---")
print(top_corr)

In [ ]:
# ============================================================
# BLOQUE 6-F: EDA — TOP 10 CORRELACIONES (SOLO TRAIN)
# ============================================================

matriz_top = df_corr_base[top_10_vars + ['POBRE']].corr(method='spearman')

plt.figure(figsize=(14, 12))
sns.heatmap(matriz_top, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=1, cbar_kws={'label': 'Coeficiente de Correlacion de Spearman'},
            annot_kws={'size': 11, 'weight': 'bold'})
plt.title('Zoom: Top 10 Variables con Mayor Relacion con la Pobreza (Train)', fontsize=16, pad=20)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 6-G: EDA — PILARES DE LA BRECHA DIGITAL (SOLO TRAIN)
# ============================================================

pilares = ['Brecha de Internet por Costo', 'Brecha de Internet por Cobertura', 'Brecha de Internet por Habilidad']
resumen_pilares = []

for pilar in pilares:
    if pilar in df_train_viz.columns:
        pct = df_train_viz.groupby('POBRE')[pilar].mean() * 100
        if 0 in pct.index:
            resumen_pilares.append({'Pilar': pilar.replace('Brecha de Internet por ', ''), 'Condicion': 'No Pobre (0)', 'Porcentaje': pct[0]})
        if 1 in pct.index:
            resumen_pilares.append({'Pilar': pilar.replace('Brecha de Internet por ', ''), 'Condicion': 'Pobre (1)',    'Porcentaje': pct[1]})

df_pilares_plot = pd.DataFrame(resumen_pilares)
plt.figure(figsize=(12, 8))
ax = sns.barplot(x='Pilar', y='Porcentaje', hue='Condicion', data=df_pilares_plot, palette=['#34495E', '#E74C3C'])
plt.title('Incidencia de los Pilares de la Brecha Digital — Train', fontsize=16, pad=20)
plt.ylabel('Hogares Afectados (%)', fontsize=12)
plt.ylim(0, 100)
for p in ax.patches:
    h = p.get_height()
    if h > 0:
        ax.annotate(f'{h:.1f}%', (p.get_x() + p.get_width()/2., h),
                    ha='center', va='bottom', fontsize=11, fontweight='bold', xytext=(0,5), textcoords='offset points')
plt.legend(title='Estado Socioeconomico')
plt.grid(axis='y', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 6-H: EDA — BOXPLOTS COMPARATIVOS (SOLO TRAIN)
# ============================================================

# Crear variable Diversidad Dispositivos en train
columnas_equipos = ['Computador de Escritorio', 'Computador Portatil', 'Tableta', 'Telefono Celular', 'Televisor inteligente']
cols_presentes = [c for c in columnas_equipos if c in df_train_viz.columns]
df_train_viz['Diversidad Dispositivos'] = df_train_viz[cols_presentes].fillna(0).sum(axis=1)

col_gasto = 'Valor Pagado Servicio Telefonia'
# Usar solo filas con dato original (no imputado) para evitar distorsion en boxplot
if col_gasto in df_nulos_snapshot.columns:
    mask_real = ~df_nulos_snapshot[col_gasto]
    df_gasto_clean = df_train_viz[mask_real & (df_train_viz[col_gasto] > 0)].copy()
else:
    df_gasto_clean = df_train_viz[df_train_viz[col_gasto] > 0].copy()

p95_gasto = df_gasto_clean[col_gasto].quantile(0.95)
df_gasto_plot = df_gasto_clean[df_gasto_clean[col_gasto] <= p95_gasto]
print(f"Hogares con gasto real reportado (train): {len(df_gasto_clean):,}")

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 7))
paleta = ['#34495E', '#E74C3C']

sns.boxplot(x='POBRE', y=col_gasto, data=df_gasto_plot, palette=paleta, ax=ax1, showfliers=False)
ax1.set_title('A. Gasto Mensual en Telefonia', fontsize=14, fontweight='bold')
ax1.set_ylabel('Pesos Colombianos (COP)', fontsize=12)
ax1.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

sns.boxplot(x='POBRE', y='Frecuencia Uso Int', data=df_train_viz, palette=paleta, ax=ax2, showfliers=False)
ax2.set_title('B. Intensidad de Uso (Frecuencia)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Escala DANE (1-5)', fontsize=12)

sns.boxplot(x='POBRE', y='Diversidad Dispositivos', data=df_train_viz, palette=paleta, ax=ax3, showfliers=False)
ax3.set_title('C. Diversidad de Dispositivos', fontsize=14, fontweight='bold')
ax3.set_ylabel('Cantidad de Equipos', fontsize=12)

for ax in [ax1, ax2, ax3]:
    ax.set_xlabel('Hogar Pobre?', fontsize=11)
    ax.set_xticks(ax.get_xticks())                              # fija posiciones antes de etiquetar
    ax.set_xticklabels(['No Pobre (0)', 'Pobre (1)'])
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.suptitle('Analisis Comparativo de la Brecha Digital en Colombia — Train', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

# Histogramas
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(22, 6))

sns.histplot(data=df_gasto_plot, x=col_gasto, hue='POBRE', element='step', kde=True,
             palette=paleta, ax=ax1, common_norm=False)
ax1.set_title('Distribucion del Gasto Mensual', fontsize=13, fontweight='bold')
ax1.xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

sns.histplot(data=df_train_viz, x='Frecuencia Uso Int', hue='POBRE', element='bars', kde=False,
             palette=paleta, ax=ax2, discrete=True, common_norm=False)
ax2.set_title('Distribucion de Intensidad de Uso', fontsize=13, fontweight='bold')

sns.histplot(data=df_train_viz, x='Diversidad Dispositivos', hue='POBRE', element='bars', kde=False,
             palette=paleta, ax=ax3, discrete=True, common_norm=False)
ax3.set_title('Distribucion de Diversidad de Equipos', fontsize=13, fontweight='bold')

for ax in [ax1, ax2, ax3]:
    ax.set_ylabel('Densidad / Frecuencia Relativa')
    leg = ax.get_legend()
    if leg:
        leg.set_title('Es Pobre?')
        for t, l in zip(leg.get_texts(), ['No Pobre (0)', 'Pobre (1)']): t.set_text(l)

plt.suptitle('Analisis de Frecuencias y Densidad de la Brecha Digital — Train', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 6-I: EDA — CORRELACIONES POR GRUPO (SOLO TRAIN)
# ============================================================

def corr_sin_constantes(df_grupo, cols):
    cols_validas = [c for c in cols if c in df_grupo.columns and df_grupo[c].std() > 0]
    return df_grupo[cols_validas].corr(method='spearman'), cols_validas

df_pobres_g    = df_train_viz[df_train_viz['POBRE'] == 1]
df_no_pobres_g = df_train_viz[df_train_viz['POBRE'] == 0]

corr_pobres,    cols_p  = corr_sin_constantes(df_pobres_g,    top_10_vars)
corr_no_pobres, cols_np = corr_sin_constantes(df_no_pobres_g, top_10_vars)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))

sns.heatmap(corr_no_pobres, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax1, cbar=False, annot_kws={'size': 9})
ax1.set_title('A. Estructura Digital: Hogares NO Pobres (Train, Spearman)', fontsize=14, fontweight='bold', pad=20)

sns.heatmap(corr_pobres, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=ax2, annot_kws={'size': 9})
ax2.set_title('B. Estructura Digital: Hogares POBRES (Train, Spearman)', fontsize=14, fontweight='bold', pad=20)

for ax in [ax1, ax2]:
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')    # rotar sin re-asignar labels

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 7: PREPROCESAMIENTO — AJUSTADO SOLO CON DATOS DE TRAIN
# Correccion critica: fit SOLO en train, transform en ambos.
# Para modelos sensibles a escala (LR, KNN) se aplica StandardScaler.
# ============================================================

# 1. Identificar tipos de columna segun distribucion en TRAIN
binary_cols  = [c for c in X_train.columns
                if set(X_train[c].dropna().unique()).issubset({0.0, 1.0})]
ordinal_cols = [c for c in X_train.columns if c not in binary_cols]

print(f"Variables binarias (0/1)  : {len(binary_cols)}")
print(f"Variables ordinales/cont. : {len(ordinal_cols)}")

# 2. Ajustar imputadores SOLO con X_train
imp_binary  = SimpleImputer(strategy='constant', fill_value=0)
imp_ordinal = SimpleImputer(strategy='median')

X_train_proc = X_train.copy()
X_test_proc  = X_test.copy()

if binary_cols:
    X_train_proc[binary_cols] = imp_binary.fit_transform(X_train[binary_cols])
    X_test_proc[binary_cols]  = imp_binary.transform(X_test[binary_cols])   # solo transform

if ordinal_cols:
    X_train_proc[ordinal_cols] = imp_ordinal.fit_transform(X_train[ordinal_cols])
    X_test_proc[ordinal_cols]  = imp_ordinal.transform(X_test[ordinal_cols])  # solo transform

# 3. Estandarizacion para Regresion Logistica y KNN
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_proc),
    columns=X_train_proc.columns, index=X_train_proc.index)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_proc),    # solo transform
    columns=X_test_proc.columns,  index=X_test_proc.index)

# 4. Verificacion
print(f"\nNulos en X_train_proc : {X_train_proc.isnull().sum().sum()}")
print(f"Nulos en X_test_proc  : {X_test_proc.isnull().sum().sum()}")
print("\nMedianas aprendidas del train (primeras 5 ordinales):")
if ordinal_cols:
    for col, med in zip(ordinal_cols[:5], imp_ordinal.statistics_[:5]):
        print(f"  {col:<40}: {med:.4f}")
print("\nLos mismos valores se aplican al test set con .transform() (sin re-fit)")

In [ ]:
# ============================================================
# BLOQUE 8: GRIDSEARCHCV — 5 MODELOS DE CLASIFICACION
# Regresion Logistica y KNN usan X_train_scaled.
# Arbol, Random Forest y XGBoost usan X_train_proc.
# Metrica de optimizacion: roc_auc (segun flujo de la profe).
# ============================================================

counts           = y_train.value_counts()
scale_pos_weight = counts[0] / counts[1]
cv               = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_cv    = {}

# ── 1. REGRESION LOGISTICA ────────────────────────────────────────────────────
# n_jobs=1 en GridSearchCV para evitar TerminatedWorkerError por memoria
# (180K filas x 62 cols: cada worker paralelo duplica la RAM requerida)
print("1/5 Regresion Logistica...")
param_lr = {'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}
grid_lr  = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    param_grid=param_lr, cv=cv, scoring='roc_auc', n_jobs=1)
grid_lr.fit(X_train_scaled, y_train)
resultados_cv['Regresion Logistica'] = {
    'auc': grid_lr.best_score_, 'params': grid_lr.best_params_,
    'model': grid_lr.best_estimator_, 'usa_scaled': True}
print(f"   AUC-ROC CV: {grid_lr.best_score_:.4f}  {grid_lr.best_params_}")

# ── 2. KNN ────────────────────────────────────────────────────────────────────
print("2/5 KNN...")
param_knn = {'n_neighbors': [11, 21, 31], 'weights': ['uniform', 'distance']}
grid_knn  = GridSearchCV(
    KNeighborsClassifier(n_jobs=1),
    param_grid=param_knn, cv=cv, scoring='roc_auc', n_jobs=1)
grid_knn.fit(X_train_scaled, y_train)
resultados_cv['KNN'] = {
    'auc': grid_knn.best_score_, 'params': grid_knn.best_params_,
    'model': grid_knn.best_estimator_, 'usa_scaled': True}
print(f"   AUC-ROC CV: {grid_knn.best_score_:.4f}  {grid_knn.best_params_}")

# ── 3. ARBOL DE DECISION ──────────────────────────────────────────────────────
print("3/5 Arbol de Decision...")
param_dt = {'max_depth': [5, 10, 15, None],
            'min_samples_split': [2, 10, 20],
            'class_weight': ['balanced']}
grid_dt  = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_dt, cv=cv, scoring='roc_auc', n_jobs=1)
grid_dt.fit(X_train_proc, y_train)
resultados_cv['Arbol de Decision'] = {
    'auc': grid_dt.best_score_, 'params': grid_dt.best_params_,
    'model': grid_dt.best_estimator_, 'usa_scaled': False}
print(f"   AUC-ROC CV: {grid_dt.best_score_:.4f}  {grid_dt.best_params_}")

# ── 4. RANDOM FOREST ─────────────────────────────────────────────────────────
print("4/5 Random Forest...")
param_rf = {'n_estimators': [100, 200], 'max_depth': [5, 10]}
grid_rf  = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    param_grid=param_rf, cv=cv, scoring='roc_auc', n_jobs=1)
grid_rf.fit(X_train_proc, y_train)
resultados_cv['Random Forest'] = {
    'auc': grid_rf.best_score_, 'params': grid_rf.best_params_,
    'model': grid_rf.best_estimator_, 'usa_scaled': False}
print(f"   AUC-ROC CV: {grid_rf.best_score_:.4f}  {grid_rf.best_params_}")

# ── 5. XGBOOST ────────────────────────────────────────────────────────────────
print("5/5 XGBoost...")
param_xgb = {
    'n_estimators':     [200, 300],
    'learning_rate':    [0.05, 0.1],
    'max_depth':        [5, 6],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3],
}
grid_xgb = GridSearchCV(
    XGBClassifier(random_state=42, objective='binary:logistic', eval_metric='logloss',
                  scale_pos_weight=scale_pos_weight, n_jobs=-1),
    param_grid=param_xgb, cv=cv, scoring='roc_auc', n_jobs=1)
grid_xgb.fit(X_train_proc, y_train)
resultados_cv['XGBoost'] = {
    'auc': grid_xgb.best_score_, 'params': grid_xgb.best_params_,
    'model': grid_xgb.best_estimator_, 'usa_scaled': False}
print(f"   AUC-ROC CV: {grid_xgb.best_score_:.4f}  {grid_xgb.best_params_}")

print("\nGridSearchCV completado para los 5 modelos.")

In [ ]:
# ============================================================
# BLOQUE 9: COMPARACION Y SELECCION DEL MODELO CAMPEON
# ============================================================

df_comparacion = pd.DataFrame([
    {'Modelo': nombre, 'AUC-ROC (CV)': info['auc'], 'Mejores Hiperparametros': str(info['params'])}
    for nombre, info in resultados_cv.items()
]).sort_values('AUC-ROC (CV)', ascending=False).reset_index(drop=True)

print("=" * 70)
print("COMPARACION DE MODELOS — VALIDACION CRUZADA (datos de entrenamiento)")
print("=" * 70)
display(df_comparacion)

# Grafico comparativo
plt.figure(figsize=(11, 5))
colores = ['#E74C3C' if i == 0 else '#7F8C8D' for i in range(len(df_comparacion))]
bars = plt.bar(df_comparacion['Modelo'], df_comparacion['AUC-ROC (CV)'], color=colores)
y_min = df_comparacion['AUC-ROC (CV)'].min()
plt.ylim(max(0, y_min - 0.03), 1.0)
plt.title('Comparacion AUC-ROC (Validacion Cruzada) — Seleccion del Modelo Campeon', fontsize=14)
plt.ylabel('AUC-ROC')
for bar, val in zip(bars, df_comparacion['AUC-ROC (CV)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# Seleccionar campeon
winner_name  = df_comparacion.iloc[0]['Modelo']
winner_info  = resultados_cv[winner_name]
best_model   = winner_info['model']
usa_scaled   = winner_info['usa_scaled']
X_train_eval = X_train_scaled if usa_scaled else X_train_proc
X_test_eval  = X_test_scaled  if usa_scaled else X_test_proc

print(f"\nMODELO CAMPEON: {winner_name} (AUC-ROC CV = {winner_info['auc']:.4f})")
print(f"Dataset de entrada: {'estandarizado' if usa_scaled else 'sin estandarizar'}")

In [ ]:
# ============================================================
# BLOQUE 10: AJUSTE DE UMBRAL (MODELO CAMPEON)
# Se usa validacion cruzada sobre TRAIN para buscar el umbral
# que maximice F1-Score y el que logre Recall >= 80%.
# ============================================================

y_proba_cv = cross_val_predict(
    best_model, X_train_eval, y_train, cv=cv, method='predict_proba')[:, 1]

precisions, recalls, thresholds = precision_recall_curve(y_train, y_proba_cv)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)

umbral_f1_optimo = thresholds[np.argmax(f1_scores)]

idx_recall80 = np.where(recalls[:-1] >= 0.80)[0]
umbral_recall_80 = thresholds[idx_recall80[-1]] if len(idx_recall80) > 0 else 0.5

print(f"Umbral por defecto    : 0.5000")
print(f"Umbral optimo (F1)    : {umbral_f1_optimo:.4f}  -> F1 = {f1_scores.max():.4f}")
print(f"Umbral para Recall>=80%: {umbral_recall_80:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(thresholds, precisions[:-1], label='Precision', color='blue')
ax1.plot(thresholds, recalls[:-1],    label='Recall',    color='orange')
ax1.plot(thresholds, f1_scores,       label='F1-Score',  color='green')
ax1.axvline(umbral_f1_optimo, linestyle='--', color='green',  label=f'Umbral F1 optimo ({umbral_f1_optimo:.2f})')
ax1.axvline(0.5,              linestyle=':',  color='gray',   label='Umbral por defecto (0.5)')
ax1.set_xlabel('Umbral de Clasificacion')
ax1.set_ylabel('Metrica')
ax1.set_title('Precision, Recall y F1 vs Umbral (CV sobre Train)')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(recalls[:-1], precisions[:-1], color='purple', lw=2)
ax2.scatter([recalls[np.argmax(f1_scores)]],
            [precisions[np.argmax(f1_scores)]],
            color='green', s=100, zorder=5, label=f'F1 maximo (umbral={umbral_f1_optimo:.2f})')
ax2.axhline(0.80, linestyle='--', color='orange', alpha=0.5, label='Recall objetivo = 80%')
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Curva Precision-Recall (CV sobre Train)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle(f'Analisis de Umbral — Modelo Campeon: {winner_name}', fontsize=14)
plt.tight_layout()
plt.show()

umbral_final = umbral_f1_optimo
print(f"\nUmbral final seleccionado: {umbral_final:.4f} (optimiza F1-Score)")

In [ ]:
# ============================================================
# BLOQUE 11: EVALUACION FINAL EN EL CONJUNTO DE PRUEBA
# Se aplica SOLO .transform() sobre X_test_eval:
# el imputador y scaler fueron ajustados unicamente en train.
# Estas son las metricas oficiales del modelo.
# ============================================================

y_proba_test = best_model.predict_proba(X_test_eval)[:, 1]
y_pred_test  = (y_proba_test >= umbral_final).astype(int)

print("=" * 60)
print(f"EVALUACION FINAL — {winner_name}")
print(f"Umbral utilizado: {umbral_final:.4f}")
print("=" * 60)
print(classification_report(y_test, y_pred_test, target_names=['No Pobre (0)', 'Pobre (1)']))

auc_test = roc_auc_score(y_test, y_proba_test)
print(f"AUC-ROC (test set): {auc_test:.4f}")
print(f"AUC-ROC (CV train): {winner_info['auc']:.4f}")

gap = abs(winner_info['auc'] - auc_test)
if gap < 0.02:
    print(f"Diferencia CV vs Test: {gap:.4f}  -> El modelo generaliza correctamente")
else:
    print(f"Diferencia CV vs Test: {gap:.4f}  -> Revisar posible overfitting")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_test),
                       display_labels=['No Pobre (0)', 'Pobre (1)']).plot(cmap='Blues', ax=axes[0])
axes[0].set_title(f'Matriz de Confusion — {winner_name}', fontsize=13)

fpr, tpr, _ = roc_curve(y_test, y_proba_test)
axes[1].plot(fpr, tpr, color='blue', lw=2, label=f'{winner_name} (AUC = {auc_test:.4f})')
axes[1].plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Clasificador Aleatorio')
axes[1].set_xlabel('Tasa de Falsos Positivos (FPR)')
axes[1].set_ylabel('Tasa de Verdaderos Positivos (TPR)')
axes[1].set_title('Curva ROC — Evaluacion Final')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BLOQUE 12: ANALISIS DE CARACTERISTICAS RELEVANTES
# A. Feature Importances / Coeficientes
# B. Analisis SHAP (TreeExplainer o LinearExplainer segun modelo)
# ============================================================

# A. Feature Importances
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X_train_proc.columns)
    top15 = importances.sort_values(ascending=False).head(15)
    plt.figure(figsize=(10, 7))
    sns.barplot(x=top15.values, y=top15.index, palette='Blues_r')
    plt.title(f'Top 15 Variables — Feature Importance ({winner_name})', fontsize=14)
    plt.xlabel('Importancia (Gini / Gain)')
    plt.tight_layout()
    plt.show()

elif hasattr(best_model, 'coef_'):
    coef_series = pd.Series(best_model.coef_[0], index=X_train_scaled.columns)
    top15_abs   = coef_series.abs().sort_values(ascending=False).head(15)
    plt.figure(figsize=(10, 7))
    sns.barplot(x=coef_series[top15_abs.index].values, y=top15_abs.index, palette='RdBu_r')
    plt.axvline(0, color='black', linewidth=1)
    plt.title(f'Top 15 Coeficientes — {winner_name}', fontsize=14)
    plt.xlabel('Coeficiente (valores positivos aumentan probabilidad de pobreza)')
    plt.tight_layout()
    plt.show()

# B. Analisis SHAP
if not SHAP_DISPONIBLE:
    print("SHAP no instalado. Ejecuta cell-00-install, reinicia el kernel y vuelve a correr.")
else:
    print("Calculando valores SHAP (muestra de 2 000 filas del train)...")
    n_muestra     = min(2000, len(X_train_eval))
    X_shap_sample = X_train_eval.sample(n_muestra, random_state=42)

    try:
        if winner_name in ['Random Forest', 'Arbol de Decision', 'XGBoost']:
            explainer  = shap.TreeExplainer(best_model)
            shap_vals  = explainer.shap_values(X_shap_sample)
            if isinstance(shap_vals, list):   # scikit-learn RF devuelve lista [clase0, clase1]
                shap_vals = shap_vals[1]
        elif winner_name == 'Regresion Logistica':
            explainer = shap.LinearExplainer(best_model, X_shap_sample)
            shap_vals = explainer.shap_values(X_shap_sample)
        else:  # KNN — KernelExplainer con submuestra reducida
            background = shap.sample(X_shap_sample, 100)
            explainer  = shap.KernelExplainer(best_model.predict_proba, background)
            shap_vals  = explainer.shap_values(X_shap_sample.iloc[:200])[:, :, 1]
            X_shap_sample = X_shap_sample.iloc[:200]

        plt.figure()
        shap.summary_plot(shap_vals, X_shap_sample, plot_type='bar', show=False)
        plt.title(f'SHAP — Importancia Global ({winner_name})', fontsize=13)
        plt.tight_layout()
        plt.show()

        plt.figure()
        shap.summary_plot(shap_vals, X_shap_sample, show=False)
        plt.title(f'SHAP — Direccion e Intensidad del Impacto ({winner_name})', fontsize=13)
        plt.tight_layout()
        plt.show()

        print("Analisis SHAP completado.")

    except Exception as e:
        print(f"Error al calcular SHAP: {e}")

In [ ]:
session_info.show()